# CIFAR-10 on the **GPU** — the GPU-resident pipeline

This notebook and its sibling [`cifar10_cpu_train.ipynb`](cifar10_cpu_train.ipynb) train the *same* resnet-18 with the *same* code (`../common/cifar_pipeline.py` + `train_engine.py`) — the **only** difference is `device`. This one runs on CUDA.

The GPU wins by keeping the whole 32x32 dataset **on the card** and augmenting there, so a fast GPU never waits on the CPU. See the CPU notebook for the mirror-image story (and why `channels_last`/`bf16` are CPU wins but not GPU wins here). The head-to-head numbers are in [`cifar10_cpu_vs_gpu.ipynb`](cifar10_cpu_vs_gpu.ipynb).

In [ ]:
# -- Shared setup: ../common has the pipeline + engine, ../a1-imagenet32 has models.py --
import os, sys
for rel in ('../common', '../a1-imagenet32'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from gpu_check import set_seed
from cifar_pipeline import load_cifar10_arrays, make_loaders
from train_engine import train
import models as M
set_seed(42)

In [ ]:
from gpu_check import get_device, enable_fast_matmul
DEVICE = get_device()
assert DEVICE.type == 'cuda', 'No GPU visible. Use cifar10_cpu_train.ipynb instead.'
enable_fast_matmul()          # TF32 + cuDNN autotune

## Data + model

`make_loaders` returns the **GPU-resident** backend for a CUDA device: the 50k images are shipped to the card once (~180 MB) and crop/flip/normalize run on the GPU in batches. `cfg` carries the measured GPU defaults (batch 512, bf16, `channels_last=False` — it's *slower* at 32x32 under this stack).

In [ ]:
trx, tryy, tex, tey = load_cifar10_arrays()
train_iter, test_iter, cfg = make_loaders(DEVICE, trx, tryy, tex, tey)
print('backend:', cfg['backend'], '| batch', cfg['batch_size'],
      '| channels_last', cfg['channels_last'], '| amp', cfg['amp_dtype'])
model = M.build('resnet18', num_classes=10)
print('resnet18 params:', sum(p.numel() for p in model.parameters()))

## Train

20 epochs gets a CIFAR-10 resnet-18 into the low-90s%. On the workstation GPU this is a couple of minutes; watch the `img/s` column.

In [ ]:
EPOCHS = 20
hist, best = train(model, train_iter, test_iter, DEVICE, epochs=EPOCHS,
                   channels_last=cfg['channels_last'], amp_dtype=cfg['amp_dtype'],
                   lr=0.1 * cfg['batch_size'] / 256, wd=5e-4)
print(f'\nBest val top-1: {best:.2%}   steady-state {sum(hist["img_s"][1:])/max(1,len(hist["img_s"])-1):,.0f} img/s')

In [ ]:
import matplotlib.pyplot as plt
xs = range(1, len(hist['top1']) + 1)
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(xs, hist['train_loss']); a.set_title('train loss'); a.set_xlabel('epoch')
b.plot(xs, hist['top1'], label='top-1'); b.plot(xs, hist['top5'], label='top-5')
b.set_title('CIFAR-10 resnet18 (GPU)'); b.set_xlabel('epoch'); b.legend()
plt.tight_layout(); plt.show()